<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 09 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Observe Sharding, Replication, and Distributed Execution</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Start with three Frontends and two Backends, expand to a third BE and watch existing replicas move onto it, prove why a direct FE connection is not a load balancer, add a health-checked HAProxy entry point, trace one aggregate across the BE scan tasks selected at runtime, observe a returning replica catch up after a write, and compare Bucket Shuffle with Colocate Join.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · FE election · HAProxy · Bucket · Tablet · Replica balancing · Query Profile · Replica version · Bucket Shuffle · Colocate</span>
</div>

**Module 9: Sharding, Replication, and Distributed Execution in Apache Doris** · [Lesson](course.md) · [Quiz](quiz09_distribution_and_availability.ipynb)

Plan about 70 minutes, a little more when its first data load runs over a slow network.

Run the cells in order. This Lab operates only the isolated `doris-rta-m09` Compose project and `m09_*` objects. It reads one bounded interval from the course's public S3 bucket, which needs no credentials, and never modifies S3. Sections 1–3 cover BE expansion, FE routing, distributed scan evidence, BE recovery, and a Join-distribution comparison.

### Initialize the Lab

Run the next cell before Section 1 and after every kernel restart. It finds the course root, loads the shared `DorisLab` helper, and records the Compose and HAProxy endpoints. It does not start or stop containers.

The helper starts with the direct `fe-1` endpoint only so it can inspect an existing Module 9 project. Later cells deliberately reconnect through the current Master, HAProxy, and the FE that coordinates Profile capture.


In [ ]:
from pathlib import Path
import shlex
import sys

import pandas as pd

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab
from doris_course.environment import (
    M09_BE_HTTP_PORT,
    M09_FE_HTTP_PORTS,
    M09_FE_QUERY_PORTS,
    M09_PROXY_PORT,
    M09_PROXY_STATS_PORT,
)
from doris_course.ui import card, show_frame

MODULE_DIR = COURSE_ROOT / "level3/module09-sharding-replication"
COMPOSE_FILE = MODULE_DIR / "compose.multinode.yml"
# Host ports: FE query 19130-19132, FE HTTP 18130-18132, HAProxy 19230 and 18230,
# unless DORIS_RTA_M09_* environment variables set others (see doris_course/environment.py).
PROXY_STATS_URL = f"http://127.0.0.1:{M09_PROXY_STATS_PORT}/stats;csv"
PROXY_CONTAINER = "doris-rta-m09-haproxy-1"
PROXY_PORT = M09_PROXY_PORT
EXPECTED_EVENTS = 12_062

FE_ENDPOINTS = {
    "172.31.91.40": {"service": "fe-1", "port": M09_FE_QUERY_PORTS["fe-1"], "http_port": M09_FE_HTTP_PORTS["fe-1"]},
    "172.31.91.41": {"service": "fe-2", "port": M09_FE_QUERY_PORTS["fe-2"], "http_port": M09_FE_HTTP_PORTS["fe-2"]},
    "172.31.91.42": {"service": "fe-3", "port": M09_FE_QUERY_PORTS["fe-3"], "http_port": M09_FE_HTTP_PORTS["fe-3"]},
}
FE1 = FE_ENDPOINTS["172.31.91.40"]
M09_HOST_PORTS = sorted({
    *M09_FE_QUERY_PORTS.values(), *M09_FE_HTTP_PORTS.values(),
    M09_BE_HTTP_PORT, M09_PROXY_PORT, M09_PROXY_STATS_PORT,
})
M09_PORT_FILTER = " ".join(f"-iTCP:{port}" for port in M09_HOST_PORTS)

lab = DorisLab(
    lab_dir=COURSE_ROOT,
    connection_host="127.0.0.1",
    connection_port=FE1["port"],
    profile_http_host="127.0.0.1",
    profile_http_port=FE1["http_port"],
);


## 1. Expand the BE layer, then recover a failed FE connection

This section follows one repeatable topology transition:

1. Start three FEs and two BEs, using a direct `fe-1` connection. HAProxy remains stopped.
2. Rebuild the two-replica Module event tables on those two live BEs.
3. Start `be-3` and observe Doris move existing replicas onto the new placement target.
4. Connect directly to the current Master, stop that FE, and observe that its existing TCP connection fails.
5. Start HAProxy without restarting the failed FE, open replacement connections through its stable port, and finally restore the stopped FE.

Stopping a container does not delete its writable layer. Rebuilding only the Module-owned event tables makes the 2 → 3 BE placement experiment repeatable without force-removing a BE.


### 1.1 Start the services with exactly two live BEs

The preparation cell follows the visual workflow used in Lab 1. Before it starts any Module 9 container, it checks that Docker has at least 4 CPUs, 8 GB of memory, and 20 GiB of free disk space, and stops with an explanation if not. It checks the isolated ports, validates the Compose file, reuses or pulls the pinned images, starts all three FEs together when persisted FE metadata requires a majority, restarts the two baseline BE processes, and explicitly keeps `be-3` and HAProxy stopped. Restarting releases process-local memory retained by earlier runs without deleting the containers or their data.

The Notebook connects directly to `fe-1`. It silently checks all registered BE metadata, displays only the two live rows, and requires exactly `be-1` and `be-2`. A previously registered `be-3` may remain in FE metadata with `Alive=false`; that is different from a third running BE.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

 echo "[1/6] Verify the Docker client, daemon, CPUs, and memory"
 docker info >/dev/null
 docker compose version
 cpus=$(docker info --format '{{{{.NCPU}}}}')
 memory=$(docker info --format '{{{{.MemTotal}}}}')
 echo "Docker allocation: $cpus CPUs, $((memory / 1000000000)) GB memory"
 if [ "$cpus" -lt 4 ] || [ "$memory" -lt 8000000000 ]; then
   echo 'Module 9 runs three FEs and three BEs on one Docker host and needs at least 4 CPUs and 8 GB of memory.'
   echo 'Raise the limits in Docker Desktop > Settings > Resources, or free them on the Docker host, then rerun this cell.'
   exit 5
 fi

 echo "[2/6] Inspect the isolated Module 9 ports"
 if command -v lsof >/dev/null 2>&1; then
   lsof -nP {M09_PORT_FILTER} -sTCP:LISTEN || true
 fi

 echo "[3/6] Validate the course Compose file"
 docker compose -f "$COMPOSE_FILE" config --quiet
 if [ "$(docker inspect -f '{{{{.State.Running}}}}' doris-rta 2>/dev/null || true)" = "true" ]; then
   echo 'The single-node doris-rta sandbox from Modules 1-8 is running. Stop it first with: docker stop doris-rta'
   exit 4
 fi

 echo "[4/6] Reuse or pull the pinned images"
 docker image inspect apache/doris:all-in-one-4.1.3 >/dev/null 2>&1 || docker pull apache/doris:all-in-one-4.1.3
 docker image inspect haproxy:3.0.30-alpine3.24 >/dev/null 2>&1 || docker pull haproxy:3.0.30-alpine3.24

 echo "[5/6] Check free disk space for container data"
 free_kb=$(docker run --rm --entrypoint df apache/doris:all-in-one-4.1.3 -Pk / | awk 'NR == 2 {{print $4}}')
 echo "Free disk space for containers: $((free_kb / 1048576)) GiB"
 if [ "$free_kb" -lt $((20 * 1048576)) ]; then
   echo 'Module 9 needs at least 20 GiB of free disk space for container data.'
   echo 'Remove unused images and stopped containers (docker system df shows the usage) or enlarge the Docker disk, then rerun this cell.'
   exit 6
 fi

 echo "[6/6] Start three FEs with freshly restarted baseline BEs; keep HAProxy stopped"
 # Reset BE process memory retained by earlier runs. `stop` preserves each
 # container writable layer and all Doris data; the following `up` starts only be-1 and be-2.
 docker compose -f "$COMPOSE_FILE" stop be-1 be-2 be-3 haproxy >/dev/null 2>&1 || true
 if docker container inspect doris-rta-m09-fe-1-1 doris-rta-m09-fe-2-1 doris-rta-m09-fe-3-1 >/dev/null 2>&1; then
   # Existing FE metadata requires a majority. Start all FE containers concurrently,
   # then let Compose wait for health and start the two baseline BEs.
   docker start doris-rta-m09-fe-1-1 doris-rta-m09-fe-2-1 doris-rta-m09-fe-3-1 >/dev/null
 fi
 docker compose -f "$COMPOSE_FILE" up -d --wait fe-1 fe-2 fe-3 be-1 be-2
 docker compose -f "$COMPOSE_FILE" ps
""", title="Prepare the two-BE Module 9 environment");

assert not lab.container_running(PROXY_CONTAINER), (
    "HAProxy must be stopped after Section 1.1; inspect the Module 9 Compose project before continuing."
)

lab.connect(
    container="doris-rta-m09-fe-1-1",
    host="127.0.0.1", port=FE1["port"], http_port=FE1["http_port"],
)
lab.execute("CREATE DATABASE IF NOT EXISTS doris_course")
lab.execute("USE doris_course")

registered_backends = lab.query("SHOW BACKENDS")
expected_registered = len(registered_backends.index)
assert expected_registered in (2, 3)
backend_state = lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=expected_registered, expected_alive=2,
    title="BE metadata with exactly two live nodes", show=False,
)
live_backends = backend_state.loc[
    backend_state["Alive"].astype(str).str.lower() == "true",
    ["BackendId", "Host", "Alive", "TabletNum", "Version"],
].reset_index(drop=True)
show_frame("Exactly two BEs are alive before expansion", live_backends)
assert set(live_backends["Host"].astype(str)) == {"172.31.91.50", "172.31.91.51"}
assert live_backends["Version"].nunique() == 1

lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    expected_roles=("FOLLOWER",), require_same_version=True,
    title="Three joined FEs and the current Master",
);


**Expected result**

- Three FEs are alive and exactly one reports `IsMaster=true`.
- The final live-BE table contains exactly two rows: `172.31.91.50` and `172.31.91.51` in this fixed Compose network.
- On a rerun, `SHOW BACKENDS` may also contain the registered `be-3` with `Alive=false`. It is not counted as a live placement target while the baseline tables are rebuilt.
- `be-1` and `be-2` start as fresh processes, so process-local memory from earlier runs is released. No BE registration, container writable layer, Doris data, or S3 object is deleted.


### 1.2 Add `be-3` and observe automatic replica migration

The event table uses two Partitions, six Buckets per Partition, and two replicas per Tablet. With only two live BEs, every logical Tablet must have one copy on each live BE. The Lab rebuilds only this Module-owned event table so every run begins from that controlled physical placement.

If a complete `m09_events` table already exists, it is temporarily renamed and used as the local 12,062-row source. The first run reads the same bounded S3 interval. After the event table is healthy on two BEs, the Lab starts `be-3` without issuing any replica-move command.


#### Rebuild and load the controlled two-BE baseline

One cell performs the complete preparation: it silently checks whether a committed local baseline can be reused, creates the two-replica `m09_events` table while only two BEs are alive, and loads it. The first run reads the bounded S3 interval; a rerun copies the local baseline and does not read S3 again. The interval's 12,062 rows are spread over all 40 course files, so Doris reads about 59 MB to find them: on a slow network the first run of this cell can take more than half as long as Lab 1's load.

The final assertions check the known row count and two-replica definition. The next cell waits for Tablet metadata and shows the compact replica placement needed for the expansion experiment.

In [ ]:
# Reuse a complete local baseline on reruns without displaying an intermediate table.
def complete_local_baseline(table_name):
    if not lab.table_exists(table_name):
        return False
    rows = lab.query(f"SELECT COUNT(*) AS row_count FROM {table_name}")
    return int(rows.iloc[0]["row_count"]) == EXPECTED_EVENTS

current_ready = complete_local_baseline("m09_events")
previous_ready = complete_local_baseline("m09_events_previous")

if current_ready:
    lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE")
    lab.execute("ALTER TABLE m09_events RENAME m09_events_previous")
    source_table = "m09_events_previous"
elif previous_ready:
    lab.execute("DROP TABLE IF EXISTS m09_events FORCE")
    source_table = "m09_events_previous"
else:
    lab.execute("DROP TABLE IF EXISTS m09_events FORCE")
    lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE")
    source_table = None

# Remove a skew-comparison table that older copies of this Lab created.
lab.execute("DROP TABLE IF EXISTS m09_events_skewed FORCE")
for table in ("m09_availability_events", "m09_availability_probe", "m09_users"):
    lab.execute(f"DROP TABLE IF EXISTS {table} FORCE")

lab.execute("""
CREATE TABLE m09_events (
    event_time DATETIME,
    event_id BIGINT,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) (
    PARTITION p_first_half VALUES LESS THAN ("2020-03-01 03:30:00"),
    PARTITION p_second_half VALUES LESS THAN ("2020-03-01 07:00:00")
)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES ("replication_num" = "2")
""")

if source_table is None:
    event_insert = """
    INSERT INTO m09_events
    SELECT event_time, event_id, user_id, product_id, event_type, region, revenue
    FROM S3(
        "uri" = "s3://doris-courses-public-dataset/doris-course/events/v1/events-*.parquet",
        "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
        "s3.region" = "us-east-1",
        "s3.credentials_provider_type" = "ANONYMOUS",
        "s3.connection.timeout" = "10000",
        "s3.connection.request.timeout" = "30000",
        "format" = "parquet"
    )
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 07:00:00'
    """
else:
    event_insert = f"""
    INSERT INTO m09_events
    SELECT event_time, event_id, user_id, product_id, event_type, region, revenue
    FROM {source_table}
    """

loaded_events = lab.insert(
    event_insert,
    title=("Load the bounded S3 events" if source_table is None else "Rebuild events from the committed local baseline"),
    expected_rows=EXPECTED_EVENTS,
    show=False,
)
assert loaded_events == EXPECTED_EVENTS
assert lab.table_replication_num("m09_events") == 2

In [ ]:
lab.wait_for_tablet_rows("m09_events", expected_rows=EXPECTED_EVENTS)
lab.wait_for_replica_health("m09_events", expected_alive=2, expected_complete=2)

before_events = lab.replica_distribution_summary("m09_events", show=False)

# The two-BE baseline must begin with 12 + 12 + 0 physical replicas.
def replica_count(frame, host):
    match = frame.loc[frame["host"].astype(str) == host, "stored replicas"]
    return int(match.iloc[0]) if not match.empty else 0

counts = [replica_count(before_events, host) for host in ("172.31.91.50", "172.31.91.51", "172.31.91.52")]
assert counts == [12, 12, 0]

before_stage = pd.DataFrame([{
    "stage": "Before be-3 starts",
    "be-1": replica_count(before_events, "172.31.91.50"),
    "be-2": replica_count(before_events, "172.31.91.51"),
    "be-3": replica_count(before_events, "172.31.91.52"),
    "total replicas": int(before_events["stored replicas"].astype(int).sum()),
}])
show_frame("Event-table replica placement before expansion", before_stage)
lab.show_statements("SHOW REPLICA DISTRIBUTION FROM m09_events", "SHOW BACKENDS")
lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE");

**Expected result**

- The displayed row describes the physical replica placement for `m09_events`.
- There are 12 logical Tablets and two replicas per Tablet, so the table has 24 physical replicas.
- While only `be-1` and `be-2` are alive, each stores one replica of every Tablet: `12 + 12 + 0 = 24`.
- This is the fixed starting point. The next cell starts `be-3` and polls Doris metadata until the new BE receives existing replicas.


#### Start `be-3` and wait for automatic replica migration

Starting `be-3` only adds a live placement target; no table is reloaded and no replica-move command is issued. The helper checks replica metadata once per second, for at most three minutes, until all three BEs store at least one replica and the total remains 24. The result is then placed beside the saved “before” row.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" up -d --wait be-3
""", title="Start the third BE")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3, require_same_version=True,
    title="Three live BEs after expansion", show=False,
)
lab.wait_for_backend_storage(expected_backends=3, show=False)

expanded_events = lab.wait_for_replica_distribution(
    "m09_events",
    expected_backends=3, expected_total_replicas=24,
    timeout_seconds=180, show=False,
)

counts = [replica_count(expanded_events, host) for host in ("172.31.91.50", "172.31.91.51", "172.31.91.52")]
assert all(count > 0 for count in counts)
assert sum(counts) == 24

logical_rows = lab.query("SELECT COUNT(*) AS row_count FROM m09_events")
assert int(logical_rows.iloc[0]["row_count"]) == EXPECTED_EVENTS

after_stage = pd.DataFrame([{
    "stage": "After automatic migration",
    "be-1": replica_count(expanded_events, "172.31.91.50"),
    "be-2": replica_count(expanded_events, "172.31.91.51"),
    "be-3": replica_count(expanded_events, "172.31.91.52"),
    "total replicas": int(expanded_events["stored replicas"].astype(int).sum()),
}])
show_frame(
    "Existing event replicas move onto the new BE",
    pd.concat([before_stage, after_stage], ignore_index=True),
)
lab.show_statements("SHOW REPLICA DISTRIBUTION FROM m09_events", "SHOW BACKENDS");

**Expected result**

- The first row stays `12, 12, 0`; the second row has a positive replica count for all three BEs. Exact counts after migration can vary.
- The total remains 24 because the table still has 12 Tablets and two replicas per Tablet. Starting another BE does not change `replication_num`.
- No `ADMIN REBALANCE`, manual Tablet copy, or table reload occurs after `be-3` starts. The changed placement is direct evidence that Doris moved existing replicas in the background.
- The code verifies that `m09_events` still contains 12,062 logical rows after replica movement.


### 1.3 Break a direct FE connection, then recover through HAProxy

The first code cell only displays the current FE state. It does not raise an error, change the connection, start HAProxy, or stop a process.

The second cell owns the destructive step. Immediately before stopping anything, it requires three live FEs and confirms that the HAProxy container is stopped. It then connects directly to the current Master FE, displays the exact direct port, stops that FE, and reruns a meaningful `m09_events` count through the original connection. The expected outcome is a connection failure because another FE cannot take ownership of that TCP socket.

Only after that failure is observed does the Lab start HAProxy with `--no-deps`, so Compose cannot restart the failed FE. New connections through the stable proxy port can then reach either surviving FE.


In [ ]:
fe_state = lab.sql(
    "SHOW FRONTENDS", title="FE state before stopping the direct FE",
    columns=["Host", "Role", "IsMaster", "Join", "Alive", "Version"],
)


In [ ]:
pre_stop_state = lab.query("SHOW FRONTENDS")
pre_stop_alive = pre_stop_state["Alive"].astype(str).str.lower() == "true"
if int(pre_stop_alive.sum()) != 3:
    raise RuntimeError(
        "Stopping an FE requires three live FEs. Rerun Section 1.1 first. "
        "No FE was stopped."
    )
if lab.container_running(PROXY_CONTAINER):
    raise RuntimeError(
        "HAProxy must be stopped before the direct-connection failure test. "
        "Rerun Section 1.1 first. No FE was stopped."
    )

master_rows = pre_stop_state.loc[
    pre_stop_state["IsMaster"].astype(str).str.lower() == "true"
]
if len(master_rows.index) != 1:
    raise RuntimeError("Expected exactly one Master FE. No FE was stopped.")
master_host = str(master_rows.iloc[0]["Host"])
master_endpoint = FE_ENDPOINTS[master_host]
master_service = master_endpoint["service"]

lab.connect(
    container=f"doris-rta-m09-{master_service}-1",
    host="127.0.0.1",
    port=master_endpoint["port"],
    http_port=master_endpoint["http_port"],
)
card(
    f"127.0.0.1:{master_endpoint['port']} → {master_service}; HAProxy port {PROXY_PORT} is stopped.",
    "info", "Direct FE selected for the failure test",
)

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop {shlex.quote(master_service)}
""", title=f"Stop the directly connected FE: {master_service}")

lab.expect_current_connection_failure(
    "SELECT COUNT(*) AS event_rows FROM m09_events",
    title="The existing direct FE connection cannot migrate",
);


**Expected result**

- The first table is observational. Seeing fewer than three live FEs does not change the environment; the following cell will refuse to stop another FE.
- With three live FEs and HAProxy stopped, the second cell identifies its explicit direct FE path (`127.0.0.1:19130`–`19132` by default, or the configured override) and then stops that FE.
- Reusing the same connection for `SELECT COUNT(*) FROM m09_events` produces the compact `connection failed` result. A different FE becoming Master cannot take ownership of the existing TCP socket.


#### Start HAProxy and establish replacement connections

HAProxy now starts while the failed FE remains stopped. It health-checks all three targets and sends each **new** connection only to an `UP` FE. Four silent `CONNECTION_ID()` probes establish four distinct sessions and make the round-robin counter change visible; their connection IDs are not printed.

The HAProxy counters demonstrate connection distribution. `SHOW FRONTENDS` separately demonstrates that Doris has retained exactly one Master among the two surviving FEs.


In [ ]:
failed_services = [
    endpoint["service"]
    for endpoint in FE_ENDPOINTS.values()
    if not lab.container_running(f"doris-rta-m09-{endpoint['service']}-1")
]
if len(failed_services) != 1:
    raise RuntimeError(
        "Expected exactly one stopped FE container before starting HAProxy. "
        "Run the preceding direct-connection failure cell first."
    )
failed_service = failed_services[0]

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" up -d --wait --no-deps haproxy
""", title="Start HAProxy while the directly connected FE remains stopped")

proxy_before = lab.wait_for_proxy_backend_status(
    PROXY_STATS_URL, failed_service, "DOWN",
    title=f"HAProxy excludes the stopped target: {failed_service}", show=False,
).set_index("FE target")
connection_ids = []
for _ in range(4):
    lab.connect(
        container=PROXY_CONTAINER,
        host="127.0.0.1", port=PROXY_PORT,
        http_host="127.0.0.1", http_port=FE1["http_port"],
    )
    connection = lab.query("SELECT CONNECTION_ID() AS connection_id")
    connection_ids.append(int(connection.iloc[0]["connection_id"]))
assert len(set(connection_ids)) == 4

proxy_after = lab.proxy_backend_summary(PROXY_STATS_URL, show=False).set_index("FE target")
connection_change = proxy_after[["proxy status"]].copy()
connection_change["new connections"] = (
    proxy_after["total connections"] - proxy_before["total connections"]
)
show_frame("New connections after one FE stopped", connection_change.reset_index())
assert int(connection_change.loc[failed_service, "new connections"]) == 0
assert int(connection_change["new connections"].sum()) >= 4

lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=2, expected_masters=1,
    title="A surviving Follower has become Master",
)
proxy_result = lab.sql(
    "SELECT COUNT(*) AS event_rows FROM m09_events",
    title="Course events remain queryable through HAProxy",
)
assert int(proxy_result.iloc[0]["event_rows"]) == EXPECTED_EVENTS;


**Expected result**

- The HAProxy counter table shows at least four new connections divided between the two surviving FE targets; the stopped target has a zero delta.
- `SHOW FRONTENDS` reports two live FEs and exactly one Master. This is Doris FE election evidence; the connection distribution comes from HAProxy.
- The `m09_events` count through the HAProxy port (`19230` by default) remains 12,062 after the original direct connection failed, proving that a newly established connection can reach a surviving FE through the stable proxy endpoint.


### 1.4 Restore the stopped FE

Restart the former Master process and wait until Doris reports three live FEs and HAProxy marks the returned target `UP`. The returned FE does not have to become Master again. The final direct connection uses the actual current Master only so later Profile retrieval has a known HTTP coordinator endpoint.


In [ ]:
stopped_services = [
    endpoint["service"]
    for endpoint in FE_ENDPOINTS.values()
    if not lab.container_running(f"doris-rta-m09-{endpoint['service']}-1")
]
if len(stopped_services) > 1:
    raise RuntimeError(
        "More than one FE container is stopped. Restore the three-FE baseline from Section 1.1."
    )

if len(stopped_services) == 1:
    recovery_service = stopped_services[0]
    lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start {shlex.quote(recovery_service)}
""", title=f"Restart the stopped FE: {recovery_service}")

    recovered_proxy = lab.wait_for_proxy_backend_status(
        PROXY_STATS_URL, recovery_service, "UP", show=False,
    )
    proxy_health = recovered_proxy[["FE target", "proxy status", "last check"]].copy()
    proxy_health["last check"] = proxy_health["last check"].replace({
        "L4OK": "L4OK — TCP connection to FE port 9030 succeeded",
    })
    show_frame(
        f"HAProxy sees the restored FE as healthy: {recovery_service}",
        proxy_health,
    )
else:
    card("All three FEs are already alive; no container restart was needed.", "skip", "FE already restored")

recovered_fes = lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="All three FEs after the stopped FE returns",
)

coordinator_host = str(
    recovered_fes.loc[recovered_fes["IsMaster"] == "true", "Host"].iloc[0]
)
coordinator = FE_ENDPOINTS[coordinator_host]
lab.connect(
    container=f"doris-rta-m09-{coordinator['service']}-1",
    host="127.0.0.1", port=coordinator["port"], http_port=coordinator["http_port"],
)
lab.execute("USE doris_course");


**Expected result**

- `proxy status=UP` means the FE is eligible to receive a new client connection.
- `last check=L4OK` means HAProxy successfully opened a Layer 4 TCP connection to that FE’s MySQL query port `9030`. It does not execute SQL or identify the Master.
- This recovery table intentionally omits connection counters. Counter deltas belong to the earlier routing comparison; cumulative totals do not prove FE recovery.
- All three FEs return to `Alive=true`, while exactly one remains Master. Restarting the FE restores future availability; it does not resurrect the TCP session that failed in Section 1.3.


## 2. Observe distributed scan work and replica recovery

Section 1 established three placement targets with two replicas per event Tablet. Section 2 now separates two questions:

1. Which BE replicas actually execute one distributed event query?
2. Can a three-replica event table accept a majority write while one BE is stopped, and does the returning replica catch up?

`m09_events` answers the first question. The small Module-owned `m09_availability_events` table answers the second without changing the fixed 12,062-row event baseline.


### 2.1 Inspect the event table's physical layout

`storage_layout_summary` reports the logical Partition, Bucket, and Tablet structure plus the storage layer's current row range. It does not identify which replica a later query will read.


In [ ]:
lab.wait_for_tablet_rows("m09_events", expected_rows=12_062)
lab.storage_layout_summary(["m09_events"]);


**Expected result**

- The single row reports two Partitions, six Buckets per Partition, 12 logical Tablets, two replicas per Tablet, and 12,062 logical rows.
- `rows / tablet` is a range across the 12 logical Tablets. It describes data placement, not the number of rows that each BE will scan in the next query.
- Two replicas per Tablet means Doris has two possible physical copies to read. A query selects one copy of each required Tablet; it does not scan both copies.


### 2.2 Observe one aggregate across the selected BE scan tasks

The query calculates eight regional totals from `m09_events`. `EXPLAIN` shows the planned fragments, Exchanges, and selected Tablet scope. The cropped `DetailProfile` keeps the BE host, Tablet IDs, and `ScanRows` for every scan task that actually ran. For this small teaching query, the Lab sets the per-Fragment Pipeline task parallelism inside each BE to `1`. This reduces repeated task instances of the same Pipeline so the output emphasizes participating BE hosts. It does not limit the query to one BE or make the whole query single-threaded.

The helper temporarily enables Profile level 2 and disables SQL Cache, Query Cache, and Condition Cache. It restores the session settings afterward. Storage and operating-system caches are not flushed, so this is not a cold-cache timing benchmark.


In [ ]:
regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m09_events
GROUP BY region
ORDER BY region
"""
regional_evidence = lab.capture_query(
    regional_query, settings={"parallel_pipeline_task_num": "1"}
)
regional_evidence.show_result("Regional totals from m09_events")
regional_evidence.show_distributed_plan("Planned fragments and scan scope")
regional_evidence.show_distributed_scan_profile("Actual BE scan tasks selected for this query")

assert int(regional_evidence.rows["event_count"].sum()) == 12_062


**Expected result**

First confirm the business answer. The result contains eight region rows, and their `event_count` values sum to 12,062.

Read the cropped `EXPLAIN` as planned work. Its structure resembles:

```text
PLAN FRAGMENT 0
  PARTITION: UNPARTITIONED
  VRESULT SINK
  VMERGING-EXCHANGE

PLAN FRAGMENT 1
  PARTITION: HASH_PARTITIONED: region[...]
  VAGGREGATE (merge finalize)
  VEXCHANGE

PLAN FRAGMENT 2
  PARTITION: RANDOM
  STREAM DATA SINK
    HASH_PARTITIONED: region[...]
  VAGGREGATE (update serialize)
  VOlapScanNode
    TABLE: doris_course.m09_events(m09_events)
    partitions=<selected>/<total>
    tablets=<selected>/<total>
    numNodes=<planned BE count>
```

`PLAN FRAGMENT 2` scans the selected Tablets and performs partial aggregation. `VEXCHANGE` transfers partial regional groups, and the upper aggregate finishes the answer. `tablets=12/12` refers to 12 logical Tablets, not both physical replicas of each Tablet.

Then read the cropped DetailProfile as actual execution evidence:

```text
Fragment 2:
  Pipeline 0(host=TNetworkAddress(hostname:<BE host>, port:9050)):
    PipelineTask(index=<task index>):
      OLAP_SCAN_OPERATOR(... table_name=m09_events(m09_events) ...):
        - TabletIds: [<selected Tablet replica IDs>]
        - ScanRows: <rows scanned by this task>
```

Focus on these fields:

- Each distinct `host=TNetworkAddress(...)` is a BE that actually executed a scan for this query. Different BE addresses show that the query was distributed across multiple BEs, whose assigned fragment instances can execute concurrently.
- `TabletIds` identifies the logical Tablet work assigned to a task. Doris chooses one available replica for each required Tablet; it does not read both replicas.
- `ScanRows` is the actual number of rows scanned by that task.
- A BE absent from this Profile did not scan `m09_events` in this execution, even if `SHOW REPLICA DISTRIBUTION` showed that it stores some replicas. In the saved run, `.50` and `.51` participated while `.52` did not.

The exact participating BE set, Tablet IDs, task grouping, and row counts may change after another replica placement or scheduling decision. The Lab therefore does not require all three BEs to appear and does not use elapsed time to claim scaling. `EXPLAIN` and this Profile also do not identify which FE planned the query; Section 1 uses the connection and HAProxy evidence for FE routing.


### 2.3 Write an event with one BE stopped

`m09_events` has two replicas. Under Doris's default majority-write rule, a two-replica table requires both copies, so they are not suitable for proving a successful write with one BE unavailable.

`m09_availability_events` therefore has one logical Tablet and three replicas. The first code cell inserts one baseline event while every BE is alive and verifies that each BE has a current copy. The next cell stops `be-2`; `be-1` and `be-3` then form a majority and commit one known purchase event without changing the fixed 12,062-row event baseline.


In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_availability_events FORCE")
lab.execute("""
CREATE TABLE m09_availability_events (
    event_id BIGINT,
    event_time DATETIME,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
UNIQUE KEY(event_id)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "3",
    "enable_unique_key_merge_on_write" = "true"
)
""")
lab.insert(
    """
    INSERT INTO m09_availability_events
    VALUES (9000001, '2020-03-01 07:00:00', 700001, 1001001, 'view', 'region_01', 0.00)
    """,
    title="Create the initial event while all three BEs are alive",
    expected_rows=1,
)
lab.wait_for_replica_health(
    "m09_availability_events", expected_alive=3, expected_complete=3,
)
availability_before = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Before the stop: every BE has the current event-table version",
)
assert len(availability_before.index) == 3
assert (availability_before["current-version replicas"] == 1).all();


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop be-2
""", title="Stop be-2")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=2,
    title="FE metadata after be-2 stops",
)
lab.execute("""
INSERT INTO m09_availability_events
VALUES (9000002, '2020-03-01 07:01:00', 700002, 1001002, 'purchase', 'region_01', 19.95)
""")
lab.sql("""
SELECT event_id, event_time, user_id, product_id, event_type, region, revenue
FROM m09_availability_events
WHERE event_id = 9000002
""", title="The purchase committed while be-2 is stopped")

availability_during = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="During the outage: be-2 retains an older event-table version",
)
assert int(availability_during["alive"].sum()) == 2
assert sorted(availability_during["current-version replicas"].tolist()) == [0, 1, 1];


**Expected result**

The business result contains the newly committed event:

```text
event_id   event_time            user_id   product_id   event_type   region      revenue
9000002    2020-03-01 07:01:00   700002    1001002      purchase     region_01   19.95
```

This proves that a purchase written after `be-2` stopped is queryable. The replica table then identifies which physical copy missed that write.

Read the replica table by comparing rows, rather than expecting fixed version numbers:

- `stored replicas` is the number of physical copies listed on that BE. It remains `1` for `be-2` because stopping a process does not delete its replica.
- `replica versions` summarizes the Doris `Version` reported by those copies. This event table has one Tablet and one copy per BE, so each row contains one version number rather than a range.
- `current-version replicas` counts copies that have reached the highest version currently reported for their Tablet and are in a successful normal state.
- The two alive BEs must have the same newer version, written here as `V_new`, and each must show one current-version replica.
- The stopped `be-2` must still have the earlier `V_old`, where `V_old < V_new`, and must show zero current-version replicas. Values such as `2` and `3` are run-specific examples; only this ordering matters.

Equal versions belong to different stages. Before the stop, all three copies are equal. Immediately after `be-2` restarts, its version may still be lower or may already equal the others if recovery finishes before the Notebook samples it. After the bounded recovery wait in Section 2.4, all three versions must be equal. While `be-2` remains stopped after the successful write, equality is not the expected result because that copy cannot receive the new version.


### 2.4 Restart the BE and wait for version recovery

`Alive=true` is only the first recovery milestone. The cell captures the replica summary immediately after the heartbeat if possible, then waits until all three copies are normal and at the target version.

Recovery may be faster than the Notebook can sample. If the immediate table already shows all copies current, that is valid evidence of fast recovery; it must not be rewritten as an observed period of lag.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start be-2
""", title="Restart be-2")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="All three BEs are alive again",
)
availability_on_return = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Event-table versions immediately after be-2 returns",
)

lab.wait_for_replica_health(
    "m09_availability_events", expected_alive=3, expected_complete=3,
    timeout_seconds=180,
)
availability_recovered = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Recovery complete: every BE has the current event-table version",
)
assert availability_recovered["alive"].all()
assert (availability_recovered["current-version replicas"] == 1).all()
lab.sql("""
SELECT event_id, event_time, user_id, product_id, event_type, region, revenue
FROM m09_availability_events
WHERE event_id = 9000002
""", title="The recovered replicas retain the committed purchase");


**Expected result**

- `SHOW BACKENDS` first confirms that all three BE processes are alive. This heartbeat evidence alone does not prove that `be-2` has repaired its replica.
- In the immediate replica table, `be-2` may still show a lower `replica versions` value and zero current-version replicas, or it may already equal the two continuously alive BEs if recovery completed before the sample.
- After `wait_for_replica_health` finishes, all three rows must report the same replica version and one current-version replica. The business query must still return event `9000002`, the `purchase` in `region_01` with revenue `19.95`.
- This proves version recovery for this controlled event table. It does not prove cross-host disaster recovery because all containers still share one Docker host and filesystem.


## 3. Reduce Join movement with Bucket Shuffle and Colocate

The event service repeatedly reports activity by user segment. The event table is already Hash distributed by `user_id`, but a small user dimension may still be Broadcast by default. This section deliberately requests a Shuffle comparison so the network requirement is visible, then changes the durable table layout and lets Doris choose Colocate.

Both experiments answer the same business question over the first event Partition:

1. summarize event rows per `user_id`;
2. join each user to `new`, `returning`, or `loyal`;
3. return event count and revenue by segment.

The per-user aggregation is part of the plan evidence. It preserves a Hash distribution on `user_id`, giving Bucket Shuffle one existing layout to reuse. This is the same kind of intermediate relation described in the Doris Bucket Shuffle documentation.

### 3.1 Reuse the event-side layout with Bucket Shuffle

The first `m09_users` definition is intentionally **not** distributed by the Join key. It uses RANDOM distribution, while the event-side intermediate result is Hash distributed by `user_id`. The `[shuffle]` hint prevents this small dimension from becoming a Broadcast comparison; it does not guarantee Bucket Shuffle, so the helper rejects the cell unless the actual Doris 4.1.3 plan reports `BUCKET_SHUFFLE`.

Removing `m09_events` from an earlier Colocation Group makes this setup rerunnable. Only the Module-owned user table is rebuilt.

In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_users FORCE")
lab.execute('ALTER TABLE m09_events SET ("colocate_with" = "")')
lab.execute("""
CREATE TABLE m09_users (
    user_id BIGINT NOT NULL,
    user_segment VARCHAR(16) NOT NULL
)
DUPLICATE KEY(user_id, user_segment)
DISTRIBUTED BY RANDOM BUCKETS 6
PROPERTIES ("replication_num" = "2")
""")

lab.insert("""
INSERT INTO m09_users
SELECT
    user_id,
    CASE MOD(user_id, 3)
        WHEN 0 THEN 'new'
        WHEN 1 THEN 'returning'
        ELSE 'loyal'
    END AS user_segment
FROM (SELECT DISTINCT user_id FROM m09_events) users
""", title="Build the user segment dimension", expected_rows=6706);

The query keeps one Partition of `m09_events`: `p_first_half`. The inner aggregation turns its event rows into one intermediate row per user. The final aggregation then restores the requested segment totals.

In [ ]:
bucket_shuffle_query = """
SELECT
    u.user_segment,
    SUM(e.event_count) AS event_count,
    SUM(e.total_revenue) AS total_revenue
FROM (
    SELECT
        user_id,
        COUNT(*) AS event_count,
        SUM(revenue) AS total_revenue
    FROM m09_events
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 03:30:00'
    GROUP BY user_id
) e
INNER JOIN [shuffle] m09_users u
    ON e.user_id = u.user_id
GROUP BY u.user_segment
ORDER BY u.user_segment
"""

bucket_shuffle_result = lab.sql(
    bucket_shuffle_query,
    title="Segment totals with the non-colocated user layout",
)
lab.explain_join_distribution(
    "EXPLAIN " + bucket_shuffle_query,
    expected_strategy="bucket_shuffle",
    title="Bucket Shuffle: Join and Exchange evidence",
);

**Expected result**

The business result contains the same three segment rows on every run of the fixed sample:

```text
user_segment   event_count   total_revenue
loyal          1098          100089.35
new            1177          113473.45
returning       994          102868.16
```

Read the cropped plan from the Join outward:

```text
VHASH JOIN
|  join op: INNER JOIN(BUCKET_SHUFFLE)
|  equal join conjunct: (user_id = user_id)
|----VEXCHANGE
...
TABLE: doris_course.m09_events(m09_events)
partitions=1/2 (p_first_half)
TABLE: doris_course.m09_users(m09_users)
```

- `partitions=1/2 (p_first_half)` proves the event predicate selected one physical Partition, which is required for this direct storage-layout reuse.
- `BUCKET_SHUFFLE` proves that the Join preserves the event-side per-user Hash layout and routes the user rows through the Join-side `VEXCHANGE` to the matching Buckets.
- Other Exchanges can still exist above the Join for the final `GROUP BY`, ordering, or result merge. The evidence concerns the data movement directly required by this Join.
- `[shuffle]` is a controlled teaching comparison. Without it, the optimizer may reasonably Broadcast this small user table.

### 3.2 Pre-position both sides for Colocate Join

A recurring event–user Join can make the compatible layout a table-design contract. The next cell rebuilds the same logical `m09_users` data with `HASH(user_id)`, six Buckets, two replicas, and the same Colocation Group as `m09_events`. It also makes `user_id` a Unique Key; every user already has one row, so the Join's input stays the same.

Doris may move replicas while the group is being formed. `IsStable=true` is therefore checked before the plan is accepted. Matching DDL alone is not enough evidence that Colocate is available.

In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_users FORCE")
lab.execute('ALTER TABLE m09_events SET ("colocate_with" = "m09_user_join_group")')
lab.execute("""
CREATE TABLE m09_users (
    user_id BIGINT NOT NULL,
    user_segment VARCHAR(16) NOT NULL
)
UNIQUE KEY(user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES (
    "replication_num" = "2",
    "enable_unique_key_merge_on_write" = "true",
    "colocate_with" = "m09_user_join_group"
)
""")

lab.insert("""
INSERT INTO m09_users
SELECT
    user_id,
    CASE MOD(user_id, 3)
        WHEN 0 THEN 'new'
        WHEN 1 THEN 'returning'
        ELSE 'loyal'
    END AS user_segment
FROM (SELECT DISTINCT user_id FROM m09_events) users
""", title="Rebuild the same users in the Colocation Group", expected_rows=6706)

lab.wait_for_colocation_group(
    "m09_user_join_group",
    title="The event-user Colocation Group is stable",
);

In [ ]:
colocate_query = """
SELECT
    u.user_segment,
    SUM(e.event_count) AS event_count,
    SUM(e.total_revenue) AS total_revenue
FROM (
    SELECT
        user_id,
        COUNT(*) AS event_count,
        SUM(revenue) AS total_revenue
    FROM m09_events
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 03:30:00'
    GROUP BY user_id
) e
INNER JOIN m09_users u
    ON e.user_id = u.user_id
GROUP BY u.user_segment
ORDER BY u.user_segment
"""

colocate_result = lab.sql(
    colocate_query,
    title="The same segment totals from the colocated layout",
)
lab.assert_same_rows(bucket_shuffle_result, colocate_result)
lab.explain_join_distribution(
    "EXPLAIN " + colocate_query,
    expected_strategy="colocate",
    title="Colocate: local Join evidence",
);

**Expected result**

The one-row group-status result must report six Buckets, two replicas, and `stable=true`. The segment totals must exactly match Section 3.1; changing data placement must not change the SQL answer.

The cropped plan should now contain:

```text
VHASH JOIN
|  join op: INNER JOIN(COLOCATE)
|  equal join conjunct: (user_id = user_id)
...
TABLE: doris_course.m09_users(m09_users)
TABLE: doris_course.m09_events(m09_events)
partitions=1/2 (p_first_half)
```

The Join fragment has no `VEXCHANGE` between the Hash Join and either input scan. Corresponding `user_id` Buckets from both tables are already placed together, so this Join does not redistribute either input. An Exchange may still appear elsewhere for the final segment aggregation or result merge.

This evidence supports a layout decision for a stable, repeated event–user relationship. It does not prove that Colocate is always faster: Bucket count controls parallelism, maintaining a Colocation Group constrains placement, and this single-host Docker cluster cannot measure production network cost.

## 4. Practice: count Tablets and replicas before you look

Section 1.2 showed that `m09_events`, with two Partitions of six Buckets and two replicas, has 12 logical Tablets and 24 replicas. Do the same arithmetic for the two other tables, before you look at their metadata:

- `m09_availability_events` from Section 2.3: no Partition clause, `DISTRIBUTED BY HASH(event_id) BUCKETS 1`, `replication_num = 3`;
- `m09_users` from Section 3.2: no Partition clause, `HASH(user_id)` with six Buckets, `replication_num = 2`.

For each table, predict the number of logical Tablets, the number of replicas, and whether two replicas of one Tablet can share a BE. Then check with `SHOW TABLETS` and count the replicas on each BE.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
from doris_course.ui import show_frame

for table, tablets, copies in (("m09_availability_events", 1, 3), ("m09_users", 6, 2)):
    replicas = lab.sql(f"SHOW TABLETS FROM {table}", title=f"Replicas of {table}",
                       columns=["TabletId", "ReplicaId", "BackendId", "Version"])
    assert replicas["TabletId"].nunique() == tablets
    assert len(replicas) == tablets * copies
    # The copies of one Tablet are on different BEs.
    assert (replicas.groupby("TabletId")["BackendId"].nunique() == copies).all()
    show_frame(f"Replicas of {table} on each BE",
               replicas.groupby("BackendId").size().rename("replicas").reset_index());

**Expected result**

| Table | Logical Tablets | Replicas | Replicas on each BE |
|---|---:|---:|---|
| `m09_availability_events` | 1 | 3 | 1 on each of the three BEs |
| `m09_users` | 6 | 12 | 4 on each of the three BEs in the tested run |

An unpartitioned table has one Partition, so its logical Tablets are its Buckets: 1 × 1 and 1 × 6. Replication multiplies them: 3 and 12 replicas. The copies of one Tablet are always on different BEs, which is why three replicas need three BEs. The checks in the solution pass silently. `TabletId`, `ReplicaId`, and `BackendId` differ on every run.

### Stop the Module 9 cluster

Run this optional cell when you want to release the three FE, three BE, and HAProxy processes. It stops the existing Module 9 containers without removing them, their writable layers, or the Doris data. Skip it if you want to continue using the cluster.

In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop
docker compose -f "$COMPOSE_FILE" ps -a
""", title="Stop the Module 9 cluster");

**Expected result:** the Module 9 services report `Exited`. Their writable layers and Doris metadata remain available.

### Restart the Module 9 cluster

Run this cell when you want to continue with the same multi-node environment. A persisted FE cluster needs a majority before any one FE can finish recovery, so the cell starts all three existing FE containers together, then starts or reuses all three BEs and HAProxy. It reconnects through the stable HAProxy port and verifies the persisted event, user, and outage-write data.

The restart cell is idempotent: it can run when the Module 9 services are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary through the shared helper; on Linux, start Docker Engine before running the cell.

In [ ]:
lab.ensure_docker_ready()

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

if docker container inspect \
  doris-rta-m09-fe-1-1 \
  doris-rta-m09-fe-2-1 \
  doris-rta-m09-fe-3-1 >/dev/null 2>&1; then
  docker start \
    doris-rta-m09-fe-1-1 \
    doris-rta-m09-fe-2-1 \
    doris-rta-m09-fe-3-1 >/dev/null
fi

docker compose -f "$COMPOSE_FILE" up -d --wait \
  fe-1 fe-2 fe-3 be-1 be-2 be-3 haproxy
""", title="Restart the Module 9 cluster")

lab.connect(
    container=PROXY_CONTAINER,
    host="127.0.0.1", port=PROXY_PORT,
    http_host="127.0.0.1", http_port=FE1["http_port"],
)
lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="Three FEs after restart", show=False,
)
lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="Three BEs after restart", show=False,
)
lab.sql("""
SELECT
    (SELECT COUNT(*) FROM m09_events) AS event_rows,
    (SELECT COUNT(*) FROM m09_users) AS user_rows,
    (SELECT COUNT(*) FROM m09_availability_events WHERE event_id = 9000002)
        AS outage_purchase_rows
""", title="Persisted Module 9 data after restart");

**Expected result:** all three FEs and BEs return to service. The persisted result reports **12,062** event rows, **6,706** user rows, and **1** purchase written during the BE outage.

## Lab complete

You separated FE election from client routing, used HAProxy to establish new connections around a stopped FE, expanded a two-BE replica layout onto a third BE, traced the BE scan tasks selected for one distributed query, committed an event through a surviving replica majority and observed the returning replica catch up, then compared Bucket Shuffle with a stable Colocate Join for the same event–user result. The stop and restart cells above preserve the isolated Module 9 cluster for later inspection.

Official references: [FE load balancing](https://doris.apache.org/docs/4.x/admin-manual/cluster-management/load-balancing/) · [Elastic scaling](https://doris.apache.org/docs/4.x/admin-manual/cluster-management/elastic-expansion/) · [Replica management](https://doris.apache.org/docs/4.x/admin-manual/maint-monitor/tablet-repair-and-balance/) · [Join strategies](https://doris.apache.org/docs/4.x/query-data/join/) · [Colocation Join](https://doris.apache.org/docs/4.x/query-acceleration/colocation-join/)

## Next

Open [Quiz 9: Sharding, Replication, and Distributed Execution in Apache Doris](quiz09_distribution_and_availability.ipynb), then continue to [Module 10: Controlling Storage Cost and Data Lifecycle in Apache Doris](../module10-managing-data/course.md).

Lab 10 uses the single-node sandbox again and stops at its first cell while the Module 9 cluster is running. Run All ends with the cluster restarted, so run **Stop the Module 9 cluster** above before you open Lab 10.